In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import numpy as np
import pandas as pd

# 1. Define the sample dataset matching our manual math
X_data = np.array([
    [1, 2],
    [2, 1]
])
y_data = np.array([6, 7])

class MultipleLinearRegressionFromScratch:
    def __init__(self):
        self.intercept_ = None
        self.coef_ = None
        self.weights_ = None

    def fit(self, X, y):
        # Step A: Add a column of 1s to X for the intercept b0
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        
        # Step B: Apply Normal Equation -> beta = (X^T * X)^(-1) * X^T * y
        XT_X = X_b.T @ X_b
        XT_y = X_b.T @ y
        
        # Solve for beta using pseudoinverse for numerical stability
        self.weights_ = np.linalg.pinv(XT_X) @ XT_y
        
        # Step C: Extract intercept (b0) and coefficients (b1, b2, ...)
        self.intercept_ = self.weights_[0]
        self.coef_ = self.weights_[1:]
        
    def predict(self, X):
        ones = np.ones((X.shape[0], 1))
        X_b = np.hstack([ones, X])
        return X_b @ self.weights_

# 2. Fit the model
model = MultipleLinearRegressionFromScratch()
model.fit(X_data, y_data)

# 3. Print computed parameters
print(f"Intercept (b0): {model.intercept_:.4f}")
print(f"Coefficients (b1, b2): {model.coef_}")

# 4. Predict on original points
predictions = model.predict(X_data)
print(f"Predictions: {predictions}")

Intercept (b0): 1.1818
Coefficients (b1, b2): [2.27272727 1.27272727]
Predictions: [6. 7.]


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Create sample dataset
data = {
    "Area": [1000, 1500, 1200, 1800, 2000, 2500, 1400, 1700],
    "Bedrooms": [2, 3, 2, 4, 4, 5, 3, 4],
    "Age": [10, 5, 8, 3, 2, 1, 7, 4],
    "Price": [200, 300, 240, 400, 450, 550, 280, 380]
}
df = pd.DataFrame(data)

# 2. Separate Features (X) and Target (y)
X = df[["Area", "Bedrooms", "Age"]]
y = df["Price"]

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Instantiate and Fit Model
model = LinearRegression()
model.fit(X_train, y_train)  # Solves OLS via SVD decomposition under the hood

# 5. Extract Learned Parameters
print("Intercept (b0):", model.intercept_)
for feature, coef in zip(X.columns, model.coef_):
    print(f"Coefficient for {feature} (b): {coef:.4f}")

# 6. Predict and Evaluate
y_pred = model.predict(X_test)

print("\n--- Evaluation Metrics ---")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"MSE:  {mean_squared_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R²:   {r2_score(y_test, y_pred):.4f}")

Intercept (b0): 152.4175824175034
Coefficient for Area (b): 0.1462
Coefficient for Bedrooms (b): 5.9341
Coefficient for Age (b): -11.9780

--- Evaluation Metrics ---
MAE:  22.0330
MSE:  542.1145
RMSE: 23.2834
R²:   0.9653


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Create a synthetic dataset with 4 features
np.random.seed(42)
n_samples = 200

house_size = np.random.normal(1800, 400, n_samples)          # Feature 1: Sq ft
num_bedrooms = np.random.randint(1, 5, n_samples)            # Feature 2: Bedrooms
age_years = np.random.uniform(1, 30, n_samples)              # Feature 3: House age
distance_city = np.random.uniform(1, 20, n_samples)          # Feature 4: Distance (miles)

# Generate target variable (Price in thousands) with known underlying weights + noise
price = (
    50                          # Base intercept
    + 0.15 * house_size         # Size contribution
    + 12.0 * num_bedrooms       # Bedroom contribution
    - 1.5 * age_years           # Age depreciation
    - 2.5 * distance_city       # Distance penalty
    + np.random.normal(0, 10, n_samples)  # Gaussian noise
)

df = pd.DataFrame({
    'House_Size': house_size,
    'Bedrooms': num_bedrooms,
    'Age': age_years,
    'Distance_To_City': distance_city,
    'Price': price
})

# 2. Separate X and y
X = df[['House_Size', 'Bedrooms', 'Age', 'Distance_To_City']]
y = df['Price']

# 3. Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Print intercept
print(f"Intercept (b0): {model.intercept_:.4f}\n")

# 6. Print each feature's coefficient
print("Feature Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  - {feature}: {coef:.4f}")

# 7. Make predictions
y_pred = model.predict(X_test)

# 8. Calculate Evaluation Metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\nModel Evaluation Metrics:")
print(f"  - MAE:  {mae:.4f}")
print(f"  - MSE:  {mse:.4f}")
print(f"  - RMSE: {rmse:.4f}")
print(f"  - R²:   {r2:.4f}")

Intercept (b0): 53.8022

Feature Coefficients:
  - House_Size: 0.1487
  - Bedrooms: 11.4086
  - Age: -1.3376
  - Distance_To_City: -2.7438

Model Evaluation Metrics:
  - MAE:  9.1732
  - MSE:  142.8622
  - RMSE: 11.9525
  - R²:   0.9535
